# 2.0版本控制台

## 正式流水线

In [ ]:
from backend2.l1.pipeline import run_l1_pipeline
from pathlib import Path
import time

# 默认只构造 h5_full 的 10%（N 轴等间隔抽样：1000 -> 100）
# 手动改为 True 可切回 100% 全构造
H5_FULL_CONSTRUCTION = False

h5_reader_cfg = {
    "kind": "h5",
    "path": "datasets/2D_rdb_NA_NA.h5",
    "dataset": "data",
    "fill_value": 0.0,
    "sample_ratio": 0.1,
    "sample_mode": "interval",
    "full_construction": H5_FULL_CONSTRUCTION,
}

configs = [
    {
        "dataset_id": "h5_full",
        "reader": h5_reader_cfg,
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "log_progress": True,
        "norm_chunk_n": 1,
    },
    {
        "dataset_id": "nc_full",
        "reader": {
            "kind": "nc",
            "path": "datasets/cylinder2d.nc",
            "var_keys": ["u", "v"],
            "time_key": "tdim",
            "y_key": "ydim",
            "x_key": "xdim",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "log_progress": True,
        "norm_chunk_n": 1,
    },
    {
        "dataset_id": "sst_full",
        "reader": {
            "kind": "mat",
            "path": "datasets/sst_weekly.mat",
            "var": "sst",
            "lon_key": "lon",
            "lat_key": "lat",
            "time_key": "time",
            "fill_value": 0.0,
        },
        "split": {
            "strategy": "temporal",
            "unit": "frame",
            "ratios": {"train": 0.8, "val": 0.1, "test": 0.1},
            "seed": 123,
        },
        "normalization": {"method": "zscore"},
        "artifacts_dir": "artifacts",
        "log_progress": True,
        "norm_chunk_n": 1,
    },
]

summaries = []
total = len(configs)
overall_t0 = time.perf_counter()

for i, cfg in enumerate(configs, start=1):
    print(f"\n[L1] ({i}/{total}) start: {cfg['dataset_id']}", flush=True)
    t0 = time.perf_counter()
    summary = run_l1_pipeline(cfg)
    dt = time.perf_counter() - t0

    l1_dir = Path(summary.artifacts_dir)
    frozen_files = {
        "array5d_norm": str(l1_dir / "array5d_norm.npy"),
        "train_split": str(l1_dir / "splits" / "train.npy"),
        "val_split": str(l1_dir / "splits" / "val.npy"),
        "test_split": str(l1_dir / "splits" / "test.npy"),
        "stats_train": str(l1_dir / "stats_train.json"),
        "manifest": str(l1_dir / "manifest.json"),
    }

    summaries.append(
        {
            "dataset_id": summary.dataset_id,
            "shape5d": list(summary.shape5d),
            "split_sizes": summary.split_sizes,
            "stats_method": summary.stats_method,
            "artifacts_dir": summary.artifacts_dir,
            "elapsed_sec": round(dt, 2),
            "frozen_files": frozen_files,
        }
    )
    print(f"[L1] ({i}/{total}) done: {summary.dataset_id} in {dt:.2f}s", flush=True)

print(f"\n[L1] all done in {time.perf_counter() - overall_t0:.2f}s", flush=True)
summaries

In [ ]:
from backend2.l2.train import run_l2_train
from backend2.l2.infer import run_l2_infer
from backend2.l2.utils import now_tag
from backend2.l2.artifact_io import ArtifactManager
from backend2.l2.data import load_l1_array_mmap, load_split_pairs, PairDataset
from torch.utils.data import DataLoader
import json

datasets = ["h5_full", "nc_full", "sst_full"]
exp_name = "baseline_unet"
sparse_input_cfg = {
    "enabled": True,
    "sample_p": 5e-3,
    "sample_sigma": 0.0,
    "sample_seed": 123,
}

all_summaries = []

for dataset_id in datasets:
    run_name = f"run_{dataset_id}_{now_tag()}"

    manager = ArtifactManager(
        artifacts_dir="artifacts",
        dataset_id=dataset_id,
        exp_name=exp_name,
        run_name=run_name,
    )

    # 1) 直接从 L1 冻结产物构建 L2 DataLoader（mmap + splits）
    array5d, manifest = load_l1_array_mmap(manager)
    train_pairs = load_split_pairs(manager, array5d, manifest, "train", target_offset=1)
    train_ds = PairDataset(
        array5d=array5d,
        pairs=train_pairs,
        target_offset=1,
        sparse_input=sparse_input_cfg,
        dataset_id=dataset_id,
    )
    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=0)
    first_batch = next(iter(train_loader))

    # 2) 训练与推理（L2.5 特征冻结默认开启）
    train_cfg = {
        "dataset_id": dataset_id,
        "artifacts_dir": "artifacts",
        "exp_name": exp_name,
        "run_name": run_name,
        "device": "auto",
        "seed": 123,
        "target_offset": 1,
        "batch_size": 8,
        "num_workers": 0,
        "epochs": 20,
        "lr": 1e-3,
        "model": {
            "base_channels": 32,
            "convs_per_stage": 2,
        },
        "sparse_input": sparse_input_cfg,
    }
    infer_cfg = {
        "dataset_id": dataset_id,
        "artifacts_dir": "artifacts",
        "exp_name": exp_name,
        "run_name": run_name,
        "device": "auto",
        "target_offset": 1,
        "batch_size": 8,
        "num_workers": 0,
        "ckpt_name": "model_best.pt",
        "freeze_features": True,
        "freeze_layers": [
            "enc.stage1.out",
            "enc.stage2.out",
            "enc.stage3.out",
        ],
        "freeze_mode": "test",
        "model": {
            "base_channels": 32,
            "convs_per_stage": 2,
        },
        "sparse_input": sparse_input_cfg,
        "probe": {
            "enabled": True,
            "record_level": 0,
            "hook_layers": ["enc.stage*.out", "dec.stage*.out", "skip.*", "head.out"],
        },
    }

    train_summary = run_l2_train(train_cfg)
    infer_summary = run_l2_infer(infer_cfg)

    all_summaries.append(
        {
            "dataset_id": dataset_id,
            "run_name": run_name,
            "loader_example": {
                "train_pairs": len(train_pairs),
                "x": list(first_batch["x"].shape),
                "y": list(first_batch["y"].shape),
            },
            "train": train_summary,
            "infer": infer_summary,
            "freeze": infer_summary.get("freeze_outputs", {}),
        }
    )

print(json.dumps(all_summaries, ensure_ascii=False, indent=2))

In [ ]:
from pathlib import Path
from backend2.l3_pre import L3PreConfig, run_l3_pre_preview

# ===== 改进版 pre_l3（FULL 数据集 × 模型列表）执行入口 =====
# 1) 对每个模型分别处理三组 full 数据集
# 2) 自动寻找每个数据集+模型实验下最新 L2 run（需存在 infer/preds_test.npz）
# 3) 输出三类 atlas 到 l3_preview 下的分类目录

# ===== 分支选择（相当于“按钮”）=====

PLOT_IO_RESIDUAL = True
PLOT_FEATURE_FLOW = True
PLOT_MODEL_KERNELS = True

# 可选模型列表：最后可只保留一个模型继续实验
MODEL_TYPES = ["unet", "unet_legacy", "vit"]
target_datasets = ["h5_full", "nc_full", "sst_full"]

artifacts_root = Path("artifacts")
BASE_EXP_NAME = "baseline_unet"
SPLIT_EXP_BY_MODEL = True


def exp_name_for_model(base_exp_name: str, model_type: str, split_exp_by_model: bool) -> str:
    if split_exp_by_model:
        return f"{base_exp_name}_{model_type}"
    return base_exp_name


def find_latest_l2_material(dataset_id: str, exp_name: str):
    exp_dir = artifacts_root / dataset_id / "L2" / exp_name
    if not exp_dir.exists():
        raise FileNotFoundError(f"未找到实验目录: {exp_dir}")

    candidates = []
    for run_dir in exp_dir.iterdir():
        if not run_dir.is_dir():
            continue
        pred_npz = run_dir / "infer" / "preds_test.npz"
        probe_summary = run_dir / "probe" / "probe_summary.json"
        if pred_npz.exists() and probe_summary.exists():
            score = max(pred_npz.stat().st_mtime, probe_summary.stat().st_mtime)
            candidates.append((score, run_dir, pred_npz, probe_summary.parent))

    if not candidates:
        # 允许没有 probe summary 的 run；若在线回退可用则会补齐代表样本层输出
        for run_dir in exp_dir.iterdir():
            if not run_dir.is_dir():
                continue
            pred_npz = run_dir / "infer" / "preds_test.npz"
            if pred_npz.exists():
                score = pred_npz.stat().st_mtime
                candidates.append((score, run_dir, pred_npz, run_dir / "probe"))

    if not candidates:
        raise FileNotFoundError(
            f"在 {exp_dir} 下没有找到 infer/preds_test.npz 的 run"
        )

    candidates.sort(key=lambda x: x[0], reverse=True)
    _, run_dir, pred_npz, probe_dir = candidates[0]
    return run_dir.name, pred_npz.resolve(), probe_dir.resolve()


l3_pre_outputs = {}
for model_type in MODEL_TYPES:
    exp_name = exp_name_for_model(BASE_EXP_NAME, model_type, SPLIT_EXP_BY_MODEL)
    l3_pre_outputs[model_type] = {}

    for dataset_id in target_datasets:
        run_name, pred_npz, probe_dir = find_latest_l2_material(dataset_id=dataset_id, exp_name=exp_name)
        out_dir = artifacts_root / dataset_id / "l3_preview" / exp_name / run_name

        cfg = L3PreConfig(
            pred_npz=str(pred_npz),
            probe_dir=str(probe_dir),
            out_dir=str(out_dir),
            sample_k=64,
            channel_k=64,
            layer_topk=6,
            sample_strategy="mixed",
            channel_select="first_k",  # 跨层共享一组 channel index，便于对照
            seed=123,
            online_fallback=True,
            device="auto",
            ckpt_name="model_best.pt",
            hook_layers=["enc.stage*.out", "dec.stage*.out", "skip.*", "head.out", "vit.block*.out", "propagator.*"],
            plot_io_residual=PLOT_IO_RESIDUAL,
            plot_feature_flow=PLOT_FEATURE_FLOW,
            plot_model_kernels=PLOT_MODEL_KERNELS,
        )

        preview_summary = run_l3_pre_preview(cfg)

        # 便于执行后快速核对三类 atlas 目录
        io_dir = artifacts_root / dataset_id / "l3_preview" / "atlas_io_residual" / exp_name / run_name
        feat_dir = artifacts_root / dataset_id / "l3_preview" / "atlas_feature_flow" / exp_name / run_name
        model_dir = artifacts_root / dataset_id / "l3_preview" / "atlas_model_kernels" / exp_name / run_name

        l3_pre_outputs[model_type][dataset_id] = {
            "model_type": model_type,
            "exp_name": exp_name,
            "run_name": run_name,
            "pred_npz": str(pred_npz),
            "probe_dir": str(probe_dir),
            "run_out_dir": str(out_dir.resolve()),
            "plot_flags": {
                "io_residual": PLOT_IO_RESIDUAL,
                "feature_flow": PLOT_FEATURE_FLOW,
                "model_kernels": PLOT_MODEL_KERNELS,
            },
            "atlas_dirs": {
                "io_residual": str(io_dir.resolve()),
                "feature_flow": str(feat_dir.resolve()),
                "model_kernels": str(model_dir.resolve()),
            },
            "outputs": preview_summary.get("outputs", {}),
            "shared_channel_indices": preview_summary.get("shared_channel_indices", []),
            "online_fallback": preview_summary.get("online_fallback", {}),
        }
        print(
            f"[l3_pre] model={model_type} dataset={dataset_id} run={run_name}\\n"
            f"  - exp_name          : {exp_name}\\n"
            f"  - plot_io_residual  : {PLOT_IO_RESIDUAL}\\n"
            f"  - plot_feature_flow : {PLOT_FEATURE_FLOW}\\n"
            f"  - plot_model_kernels: {PLOT_MODEL_KERNELS}\\n"
            f"  - atlas_io_residual : {io_dir.resolve()}\\n"
            f"  - atlas_feature_flow: {feat_dir.resolve()}\\n"
            f"  - atlas_model_kernels: {model_dir.resolve()}"
        )

l3_pre_outputs

## 旁路系统测试

In [ ]:
from backend2.sideway import run_sideway_mini
import json

# sideway 旁路单次实验：直接改这里的参数即可
MODEL_TYPE = "vit"  # 可选: "unet" | "unet_legacy" | "vit"
SAMPLE_P = 1e-2
SAMPLE_SIGMA = 0.0
SAMPLE_SEED = 123
TRAIN_STEPS = 1000
VAL_INTERVAL_STEPS = 50
BATCH_SIZE = 8
DEBUG_PLOT_SAMPLE_INDEX = 0
DEBUG_PLOT_CHANNEL = 0

sideway_result = run_sideway_mini(
    sample_p=SAMPLE_P,
    model_type=MODEL_TYPE,
    sample_sigma=SAMPLE_SIGMA,
    sample_seed=SAMPLE_SEED,
    train_steps=TRAIN_STEPS,
    val_interval_steps=VAL_INTERVAL_STEPS,
    batch_size=BATCH_SIZE,
    artifacts_dir="artifacts",
    exp_name="sideway_unet",
    force_rebuild_l1=False,
    debug_plot_sample_index=DEBUG_PLOT_SAMPLE_INDEX,
    debug_plot_channel=DEBUG_PLOT_CHANNEL,
    log_progress=True,
)

compact = []
for run in sideway_result["runs"]:
    compact.append(
        {
            "model_type": run.get("model_type"),
            "dataset_id": run["dataset_id"],
            "run_name": run["run_name"],
            "quad_plot": run.get("quad_plot", ""),
            "mse": run.get("metrics", {}).get("mse"),
            "mae": run.get("metrics", {}).get("mae"),
            "rmse": run.get("metrics", {}).get("rmse"),
        }
    )

print(json.dumps(compact, ensure_ascii=False, indent=2))

In [ ]:
from backend2.sideway import run_sideway_sample_sweep
import json

# 采样率扫描：用于定位均值坍缩阈值
MODEL_TYPE = "unet"  # 可选: "unet" | "unet_legacy" | "vit"

SAMPLE_P_LIST = [1e-2, 5e-3, 2e-3, 1e-3]
TRAIN_STEPS_PER_RUN = 500
VAL_INTERVAL_STEPS = 50

sweep_results = run_sideway_sample_sweep(
    sample_ps=SAMPLE_P_LIST,
    model_type=MODEL_TYPE,
    sample_sigma=0.0,
    sample_seed=123,
    train_steps=TRAIN_STEPS_PER_RUN,
    val_interval_steps=VAL_INTERVAL_STEPS,
    batch_size=8,
    artifacts_dir="artifacts",
    exp_name="sideway_unet_sweep",
    force_rebuild_l1=False,
    debug_plot_sample_index=0,
    debug_plot_channel=0,
    log_progress=True,
)

sweep_compact = []
for item in sweep_results:
    for run in item["runs"]:
        metrics = run.get("metrics", {})
        sweep_compact.append(
            {
                "model_type": run.get("model_type"),
                "sample_p": run.get("sample_p"),
                "dataset_id": run.get("dataset_id"),
                "run_name": run.get("run_name"),
                "mse": metrics.get("mse"),
                "mae": metrics.get("mae"),
                "rmse": metrics.get("rmse"),
                "quad_plot": run.get("quad_plot", ""),
            }
        )

print(json.dumps(sweep_compact, ensure_ascii=False, indent=2))